# Module 4: Machine Learning for Business

This final module introduces machine learning algorithms applied to business analytics — covering unsupervised learning (clustering), supervised classification, supervised regression, and model evaluation.

## Section 1: Clustering

### Objective
To apply K-Means clustering to segment Gogoro customers into distinct groups based on their construct scores, enabling targeted marketing strategies.

**Keywords:** K-Means, unsupervised learning, Elbow Method, within-cluster sum of squares (WCSS), StandardScaler, customer segmentation, centroid

### Statistical Perspective
**Clustering** is an **unsupervised machine learning** technique — no pre-defined labels are required. The algorithm discovers natural groupings in the data by minimizing within-cluster variance.

**K-Means Algorithm:**
Partitions $n$ observations into $K$ clusters by iteratively:

1. Randomly initializing $K$ cluster centroids.
2. Assigning each data point to the nearest centroid (Euclidean distance).
3. Recalculating centroids as the mean of assigned points.
4. Repeating until assignments no longer change (convergence).

**Objective Function:**

$$\underset{C}{\arg\min} \sum_{k=1}^{K} \sum_{x_i \in C_k} \| x_i - \mu_k \|^2$$

**Choosing K — The Elbow Method:**
Run K-Means for $K = 1, 2, \ldots, 10$ and plot the **Within-Cluster Sum of Squares (WCSS)**. The optimal $K$ is at the "elbow" — where WCSS decreases more slowly (diminishing returns from adding clusters).

**Feature Scaling:** K-Means uses distance, so all features must be standardized (mean = 0, SD = 1) before clustering to prevent high-variance variables from dominating.

**Industry Application:** Gogoro can use clustering to identify customer segments: e.g., *Cluster 1* = price-sensitive commuters; *Cluster 2* = eco-conscious urban riders; *Cluster 3* = brand-loyal enthusiasts. Each segment receives a different marketing message.

### Python Example Code


In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('gogoro_data.csv')

df['Brand'] = df[['Brand1', 'Brand2']].mean(axis=1)
df['Func']  = df[['Func1',  'Func2' ]].mean(axis=1)
df['Sat']   = df[['Sat1',   'Sat2'  ]].mean(axis=1)
df['Loyal'] = df[['Loyal1', 'Loyal2']].mean(axis=1)

features = df[['Brand', 'Func', 'Sat', 'Loyal']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

print(df.groupby('Cluster')[['Brand', 'Func', 'Sat', 'Loyal']].mean().round(2))



### Student Task
Create `StudentName_Clustering.py` in Visual Studio Code. Add `Price` and `Usab` construct scores to the feature set and re-run K-Means with $K = 3$. Print the cluster means for all six constructs. Write a one-sentence business description for each cluster based on their characteristic scores.

### Evaluation Questions
1. What is the difference between supervised and unsupervised machine learning?
2. How does the K-Means algorithm assign data points to clusters?
3. Why must features be standardized before applying K-Means?
4. What does the Elbow Method reveal, and how is the optimal $K$ identified?
5. How can customer clusters from K-Means inform Gogoro’s marketing strategy?

## Section 2: Classification

### Objective
To predict a binary outcome (High vs Low Loyalty) using Logistic Regression and Decision Trees, and evaluate classifier performance using precision, recall, and F1-Score.

**Keywords:** classification, logistic regression, decision tree, precision, recall, F1-Score, confusion matrix, supervised learning, binary outcome

### Statistical Perspective
**Classification** is a **supervised machine learning** task where the model learns to assign observations to predefined categories based on labeled training data.

**Common Algorithms:**

1. **Logistic Regression:** Models the probability of a binary outcome using the sigmoid function:

$$P(y=1) = \frac{1}{1 + e^{-(b_0 + b_1 x_1 + \ldots + b_k x_k)}}$$

A threshold (typically $0.50$) converts probabilities to class labels.

2. **Decision Tree:** Recursively splits the feature space using if-then rules, selecting splits that maximize class purity (Gini impurity or information gain).

**Model Evaluation Metrics:**

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| Accuracy | (TP+TN) / Total | Overall correct predictions |
| Precision | TP / (TP+FP) | Correctness of positive predictions |
| Recall | TP / (TP+FN) | Coverage of actual positives |
| F1-Score | 2·P·R / (P+R) | Harmonic mean of precision and recall |

**Train-Test Split:** Data is split (80% train / 20% test) so performance is evaluated on unseen data, preventing overfitting.

**Industry Application:** Banks classify loan applications as "approve" or "reject" based on income and credit history. Gogoro can classify customers as "likely to repurchase" or "likely to churn" based on satisfaction and brand perception scores.

### Python Example Code


In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('gogoro_data.csv')

df['Loyal_binary'] = (df[['Loyal1', 'Loyal2']].mean(axis=1) >= 4).astype(int)
features = ['Brand1', 'Brand2', 'Func1', 'Func2', 'Sat1', 'Sat2']
X = df[features]
y = df['Loyal_binary']

X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

clf = LogisticRegression(random_state=42)
clf.fit(X_train, y_train)
print(classification_report(y_test, clf.predict(X_test)))



### Student Task
Create `StudentName_Classification.py` in Visual Studio Code. Replace `LogisticRegression` with `DecisionTreeClassifier` from `sklearn.tree`. Train on the same features and print the classification report. Compare accuracy and F1-Score between both models. Which performs better on the test set?

### Evaluation Questions
1. What is the difference between classification and regression in machine learning?
2. How does Logistic Regression produce a class prediction from a probability?
3. Why is accuracy alone insufficient when classes are imbalanced?
4. What does a high Recall but low Precision indicate about a classifier?
5. Why is it essential to evaluate classification models on a held-out test set?

## Section 3: Regression

### Objective
To predict continuous outcomes (Customer Satisfaction scores) using Linear Regression, Ridge Regression, and Lasso Regression, and compare their predictive performance on unseen data.

**Keywords:** linear regression, Ridge regression, Lasso regression, regularization, RMSE, R-squared, bias-variance trade-off, overfitting, feature selection

### Statistical Perspective
**Regression** in machine learning predicts a **continuous numerical outcome** from input features. Unlike statistical regression used for hypothesis testing, ML regression focuses on **predictive accuracy on unseen data**.

**Algorithms:**

1. **Linear Regression:** Minimizes the sum of squared residuals. Best when predictors are independent and relationships are linear.

2. **Ridge Regression ($L_2$ Regularization):** Adds penalty $\lambda \sum \beta_j^2$ to the loss function, shrinking coefficients toward zero. Controls overfitting when predictors are correlated.

3. **Lasso Regression ($L_1$ Regularization):** Adds penalty $\lambda \sum |\beta_j|$, which can reduce some coefficients to exactly zero — performing automatic feature selection.

**Bias-Variance Trade-off:**

- **High Bias (Underfitting):** Model too simple; misses true patterns.
- **High Variance (Overfitting):** Model memorizes training data; fails on new data.
- Regularization reduces variance at the cost of slight bias — improving generalization.

**Evaluation Metrics:**

| Metric | Interpretation |
|--------|----------------|
| MAE | Mean absolute error; easy to interpret |
| RMSE | Penalizes large errors more heavily |
| $R^2$ | Proportion of variance explained (higher = better) |

**Industry Application:** Gogoro's sales team can predict next quarter's Customer Satisfaction scores from current brand perception and functional quality metrics, enabling proactive service interventions before loyalty declines.

### Python Example Code


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('gogoro_data.csv')
features = ['Brand1', 'Brand2', 'Func1', 'Func2',
            'Usab1', 'Usab2', 'Price1', 'Price2']
X = df[features]
y = df[['Sat1', 'Sat2']].mean(axis=1)

X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

for name, model in [('Linear', LinearRegression()),
                     ('Ridge',  Ridge(alpha=1.0)),
                     ('Lasso',  Lasso(alpha=0.1))]:
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    print(f"{name:8s}  RMSE={rmse:.3f}  R2={r2:.3f}")



### Student Task
Create `StudentName_Regression.py` in Visual Studio Code. Using the same feature set and train-test split, predict `Loyal` (average of Loyal1 and Loyal2) instead of `Sat`. Compare Linear, Ridge, and Lasso using RMSE and $R^2$ on the test set. Adjust the `alpha` hyperparameter for Ridge and Lasso and observe the effect on test performance.

### Evaluation Questions
1. What is the fundamental difference between regression for hypothesis testing and regression in machine learning?
2. How does Ridge Regression prevent overfitting compared to standard Linear Regression?
3. What unique property does Lasso Regression have that Ridge does not?
4. Why is RMSE evaluated on the test set rather than the training set?
5. In the bias-variance trade-off, where does a heavily regularized model sit, and what are the implications?

## Section 4: Model Evaluation and Selection

### Objective
To systematically compare machine learning models using cross-validation and performance metrics, and select the best model for deployment in a business context.

**Keywords:** cross-validation, k-fold CV, Grid Search, hyperparameter tuning, StratifiedKFold, model selection, generalization error, overfitting

### Statistical Perspective
Building a model is only half the job. **Model evaluation** determines whether a trained model will perform reliably on future, unseen business data. **Model selection** chooses the best algorithm and hyperparameters using principled, data-driven criteria.

**Cross-Validation (k-Fold CV):**
The dataset is split into $k$ equal folds. The model trains on $k-1$ folds and validates on the remaining fold — repeated $k$ times. The final score is the mean across all folds:

$$CV_{score} = \frac{1}{k} \sum_{i=1}^{k} \text{Score}(M, D_i)$$

**Hyperparameter Tuning — Grid Search:**
Systematically searches a predefined grid of hyperparameter values, evaluating each combination via cross-validation and selecting the best-performing combination.

**Learning Curves:**
Plot training score and CV score as a function of training set size:

- **High bias (underfitting):** Both scores are low — add complexity.
- **High variance (overfitting):** Training score is high but CV score is much lower — reduce complexity or gather more data.

**Model Comparison Framework:**

| Model | Strengths | Weaknesses |
|-------|-----------|------------|
| Logistic Regression | Interpretable, fast | Linear boundaries only |
| Decision Tree | Intuitive rules | Prone to overfitting |
| Random Forest | High accuracy, robust | Less interpretable |
| Ridge/Lasso | Handles multicollinearity | Still linear |

**Industry Application:** Gogoro's data team tests three churn prediction models side-by-side using 5-fold CV. The model with the best trade-off between F1-Score and interpretability is chosen for deployment — because executives need to understand *why* a customer is flagged as at-risk, not just *that* they are.

### Python Example Code


In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('gogoro_data.csv')

df['Loyal_binary'] = (df[['Loyal1', 'Loyal2']].mean(axis=1) >= 4).astype(int)
features = ['Brand1', 'Brand2', 'Func1', 'Func2', 'Sat1', 'Sat2',
            'Price1', 'Usab1']
X = df[features]
y = df['Loyal_binary']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=4, random_state=42),
}

for name, model in models.items():
    scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='f1')
    print(f"{name:25s}  F1 = {scores.mean():.3f} +/- {scores.std():.3f}")



### Student Task
Create `StudentName_ModelEvaluation.py` in Visual Studio Code. Add a third model — `RandomForestClassifier` from `sklearn.ensemble` (use `n_estimators=100`) — to the comparison loop. Report the 5-fold CV F1-Score for all three models. Which model would you recommend for Gogoro's churn prediction task? Justify your choice in a comment using at least two criteria (e.g., performance, interpretability, training time).

### Evaluation Questions
1. Why is k-fold cross-validation preferred over a single train-test split for model evaluation?
2. What does a large gap between training score and cross-validation score indicate?
3. How does Grid Search help in selecting the best model hyperparameters?
4. Why might a business prefer a slightly less accurate model that is more interpretable?
5. What considerations beyond accuracy matter when selecting a machine learning model for deployment?


